# Example pipeline data preparation.

## Prepare notebook.

### Import the libraries.

In [1]:
from sc_flow.data import DataManager
from sc_flow.data.sim import get_dummy_adata

### Generate dummy data.

In [2]:
BIG = False
if BIG:
    n_obs_pert = 1_000_000
    n_obs_ctrl = 500_000
else:
    n_obs_pert = 10000
    n_obs_ctrl = 5000
adata = get_dummy_adata(n_obs_pert=n_obs_pert, n_obs_ctrl=n_obs_ctrl)
adata

/Users/lorenzo.consoli/micromamba/envs/sc-flow-tools/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/Users/lorenzo.consoli/micromamba/envs/sc-flow-tools/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 15000 × 400
    obs: 'drugA', 'drugB', 'koA', 'koB', 'target', 'source_split', 'is_control'
    uns: 'drug', 'ko', 'source_split'
    obsm: 'drugA_time', 'drugA_dose', 'drugB_time', 'drugB_dose', 'koA_time', 'koA_dose', 'koB_time', 'koB_dose', 'paired_condition', 'X_repr', 'target_variable', 'X_src', 'X_tgt'

## Case 1. Loading only state data.

### Initialize data manager and get data.

In [3]:
dm = DataManager(
    sample_rep="X_repr",
)
train_collection = dm.get_matched_data(adata)
matched_data = train_collection[()][()]
matched_data

MatchedData:
 * (target) -> 	DistributionData:
	 * n_obs=15000
	 state=StateData(n_obs=15000, spatial_dims=(200,))
	 target=None
	 condition=None
	 groups=CategoricalData(n_obs=15000, n_vars=0, columns=[], repr_dict_keys=[], categorical_encoders_keys=[])
	 coupling=None

## Case 2: Grouping data based on source split.

### Initialize data manager and get data.

In [4]:
dm = DataManager(
    sample_rep="X_repr",
    groups=["source_split"],
    groups_encoding={"source_split": "one-hot"},
)
train_collection = dm.get_matched_data(adata)
matched_data = train_collection[("source_split0",)][()]
matched_data

MatchedData:
 * (target) -> 	DistributionData:
	 * n_obs=3029
	 state=StateData(n_obs=3029, spatial_dims=(200,))
	 target=None
	 condition=None
	 groups=CategoricalData(n_obs=3029, n_vars=1, columns=['source_split'], repr_dict_keys=[], categorical_encoders_keys=['source_split'])
	 coupling=None

## Case 2: Grouping data based on source split and condition.


### Initialize data manager and get data.

In [5]:
dm = DataManager(
    sample_rep="X_repr",
    conditions={
        "drug": ("drugA", "drugB"),
        "ko": ("koA", "koB"),
    },
    conditions_reps={
        "drug": "drug",
        "ko": "ko",
    },
)
train_collection = dm.get_matched_data(adata)
matched_data = train_collection[()][("drug1", "drug1", "ko1", "ko1")]
matched_data

MatchedData:
 * (target) -> 	DistributionData:
	 * n_obs=41
	 state=StateData(n_obs=41, spatial_dims=(200,))
	 target=None
	 condition=MixedTypeData(n_obs=41, categorical(n_vars=4, columns=['drugA', 'drugB', 'koA', 'koB']), continuous=None
	 groups=CategoricalData(n_obs=41, n_vars=0, columns=[], repr_dict_keys=[], categorical_encoders_keys=[])
	 coupling=None

## Case 3: Grouping data based on source split and condition with controls.


### Initialize data manager and get data.

In [6]:
dm = DataManager(
    sample_rep="X_repr",
    conditions={
        "drug": ("drugA", "drugB"),
        "ko": ("koA", "koB"),
    },
    conditions_reps={
        "drug": "drug",
        "ko": "ko",
    },
    conditions_covariates=["paired_condition"],
    groups=["source_split"],
    groups_encoding={"source_split": "one-hot"},
    control_values_dict={"drug": "control", "ko": "control"},
)
train_collection = dm.get_matched_data(adata)
matched_data = train_collection[("source_split3",)][("drug0", "drug2", "ko4", "ko0")]
matched_data

MatchedData:
 * (target) -> 	DistributionData:
	 * n_obs=8
	 state=StateData(n_obs=8, spatial_dims=(200,))
	 target=None
	 condition=MixedTypeData(n_obs=8, categorical(n_vars=4, columns=['drugA', 'drugB', 'koA', 'koB']), continuous(keys=['paired_condition'], spatial_dims={'paired_condition': 100})
	 groups=CategoricalData(n_obs=8, n_vars=1, columns=['source_split'], repr_dict_keys=[], categorical_encoders_keys=['source_split'])
	 coupling=None
 * (source) -> 	DistributionData:
	 * n_obs=1002
	 state=StateData(n_obs=1002, spatial_dims=(200,))
	 target=None
	 condition=MixedTypeData(n_obs=1002, categorical(n_vars=4, columns=['drugA', 'drugB', 'koA', 'koB']), continuous(keys=['paired_condition'], spatial_dims={'paired_condition': 100})
	 groups=CategoricalData(n_obs=1002, n_vars=1, columns=['source_split'], repr_dict_keys=[], categorical_encoders_keys=['source_split'])
	 coupling=None